# ChromaDB local use case

This notebook connects to a local Chroma server (Docker or `chroma run`) and demonstrates adding and querying data.

If you are not running a server, see the embedded alternative near the end.

## Connection settings

This notebook targets the Docker container port mapping (`localhost:8001` by default).
If your server is on a different port, set an env var before running this notebook:

```
export CHROMA_PORT=8001
```

In [1]:
import os
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
print("Heartbeat:", client.heartbeat())

Heartbeat: 1770715416660338587


In [2]:
collection = client.get_or_create_collection("local_demo")

collection.add(
    ids=["doc-1", "doc-2", "doc-3"],
    documents=[
        "Chroma is a vector database for embeddings.",
        "A vector search lets you find similar meaning.",
        "macOS users can run Chroma locally using Docker or chroma run.",
    ],
    metadatas=[
        {"source": "docs", "topic": "chroma"},
        {"source": "docs", "topic": "search"},
        {"source": "docs", "topic": "mac"},
    ],
)

print("Count:", collection.count())

/Users/tsm/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 79.3M/79.3M [00:50<00:00, 1.65MiB/s]


Count: 3


In [3]:
results = collection.query(
    query_texts=["How do I run Chroma locally on a Mac?"],
    n_results=2,
)
results

{'ids': [['doc-3', 'doc-1']],
 'distances': [[0.3023473, 1.2297055]],
 'embeddings': None,
 'metadatas': [[{'source': 'docs', 'topic': 'mac'},
   {'source': 'docs', 'topic': 'chroma'}]],
 'documents': [['macOS users can run Chroma locally using Docker or chroma run.',
   'Chroma is a vector database for embeddings.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [4]:
# Read back specific documents by id
collection.get(ids=["doc-1", "doc-3"])

{'ids': ['doc-1', 'doc-3'],
 'embeddings': None,
 'metadatas': [{'source': 'docs', 'topic': 'chroma'},
  {'topic': 'mac', 'source': 'docs'}],
 'documents': ['Chroma is a vector database for embeddings.',
  'macOS users can run Chroma locally using Docker or chroma run.'],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents']}

## Optional: Embedded mode (no server)

If you are not running a server, use the in-process client instead:

In [5]:
from chromadb import PersistentClient

embedded_client = PersistentClient(path="./chroma_data")
embedded_collection = embedded_client.get_or_create_collection("local_demo")
print("Embedded count:", embedded_collection.count())

Embedded count: 0


## Next steps

- Use your own documents and metadata fields.
- Connect this collection to your RAG pipeline or notebook experiments.